# LeetCode 3621 — Number of Integers With Popcount-Depth Equal to K I bằng Python

- Link: https://leetcode.com/problems/number-of-integers-with-popcount-depth-equal-to-k-i/
- Độ khó: **Hard**
- Chủ đề: Bit, **đếm số `≤ n` có đúng `c` bit 1** (tổ hợp theo bit), digit DP, rút gọn "độ sâu" qua popcount

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def popcountDepth(self, n: int, k: int) -> int:
```

## 1. Mục tiêu học

1. Nhận ra `depth(x) = 1 + depth(popcount(x))` (với `x > 1`) ⇒ độ sâu của `x` chỉ phụ thuộc **số bit 1** của `x`.
2. Vì `n ≤ 10^15 < 2^50`, `popcount(x) ≤ 50` ⇒ tính trước `depth(c)` cho `c ≤ 50`.
3. Đếm `#{x ≤ n : popcount(x) = c}` bằng **tổ hợp theo bit** của `n` (kỹ thuật digit DP nhị phân).
4. Xử lý trường hợp đặc biệt `x = 1` (`depth = 0`) và `k = 0`.
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Dãy `p0 = x`, `p_{i+1} = popcount(p_i)`. **Độ sâu popcount** của `x` là `d` nhỏ nhất với `p_d = 1`. Đếm số `x ∈ [1, n]` có độ sâu đúng `k`.

**Ví dụ 1:** `n = 4, k = 1` → `2` (`2`, `4`).

**Ví dụ 2:** `n = 7, k = 2` → `3` (`3`, `5`, `6`).

**Ràng buộc:** `1 ≤ n ≤ 10^15`, `0 ≤ k ≤ 5`.

In [1]:
from functools import lru_cache
from math import comb
import random, time

examples = [(4, 1, 2), (7, 2, 3)]
for n, k, exp in examples:
    print(n, k, "-> kỳ vọng", exp)

4 1 -> kỳ vọng 2
7 2 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Độ sâu popcount

In [2]:
def depth(x):
    d = 0
    while x != 1:
        x = bin(x).count("1"); d += 1
    return d

print({x: depth(x) for x in range(1, 17)})
print("độ sâu lớn nhất với c ≤ 50:", max(depth(c) for c in range(1, 51)))

{1: 0, 2: 1, 3: 2, 4: 1, 5: 2, 6: 2, 7: 3, 8: 1, 9: 2, 10: 2, 11: 3, 12: 2, 13: 3, 14: 3, 15: 2, 16: 1}
độ sâu lớn nhất với c ≤ 50: 3


### 3.2. Đếm số `≤ n` có đúng `c` bit 1

Duyệt bit của `n` từ cao xuống, giữ `ones` = số bit 1 của `n` đã đi qua. Tại một bit 1 ở vị trí `b` của `n`: nếu đặt **0** ở đây (số nhỏ hơn `n`), `b` bit thấp hơn tự do ⇒ cộng `C(b, c − ones)`. Rồi đặt 1 (bám `n`), tăng `ones`. Cuối cùng cộng 1 nếu chính `n` có đúng `c` bit 1.

In [3]:
def count_with_ones(n, c):
    if c < 0: return 0
    res = 0; ones = 0
    for b in range(n.bit_length() - 1, -1, -1):
        if n >> b & 1:
            if 0 <= c - ones <= b: res += comb(b, c - ones)
            ones += 1
    return res + (1 if ones == c else 0)

print([count_with_ones(10, c) for c in range(5)], "| kiểm tra:", [sum(1 for x in range(11) if bin(x).count("1") == c) for c in range(5)])

[1, 4, 5, 1, 0] | kiểm tra: [1, 4, 5, 1, 0]


**Lưu ý:** hàm đếm cả số `0` khi `c = 0`; trong bài ta chỉ dùng `c ≥ 1`.

### 3.3. Bảng `depth(c)` cho `c ≤ 50`

Vì `n ≤ 10^{15} < 2^{50}`, mọi `x ≤ n` có `popcount(x) ≤ 50`. Gom các `c` theo độ sâu: độ sâu 0 chỉ có `c = 1`; độ sâu 1 là các lũy thừa của 2 (`2, 4, 8, 16, 32`); độ sâu 2 là các số có popcount là lũy thừa của 2 lớn hơn 1, … Với `c ≤ 50`, độ sâu lớn nhất là 3 (số nhỏ nhất có độ sâu 4 là `127`) — nên `x ≤ 10^{15}` có độ sâu tối đa 4, và với `k = 5` đáp án luôn là 0.

In [4]:
groups = {}
for c in range(1, 51): groups.setdefault(depth(c), []).append(c)
for d in sorted(groups): print(f"depth {d}: {groups[d]}")

depth 0: [1]
depth 1: [2, 4, 8, 16, 32]
depth 2: [3, 5, 6, 9, 10, 12, 15, 17, 18, 20, 23, 24, 27, 29, 30, 33, 34, 36, 39, 40, 43, 45, 46, 48]
depth 3: [7, 11, 13, 14, 19, 21, 22, 25, 26, 28, 31, 35, 37, 38, 41, 42, 44, 47, 49, 50]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

- `k = 0`: chỉ `x = 1` ⇒ đáp án `1` (vì `n ≥ 1`).
- `k ≥ 1`: `x > 1` có độ sâu `k` ⇔ `depth(popcount(x)) = k − 1`. Lấy mọi `c ∈ [1, 50]` thoả `depth(c) = k − 1`, cộng `count_with_ones(n, c)`. Nếu `c = 1` thì phải **trừ** `x = 1` (có popcount 1 nhưng độ sâu 0).

Ví dụ 2 (`n = 7, k = 2`): cần `depth(c) = 1`, tức `c` là lũy thừa của 2 lớn hơn 1: `c ∈ {2, 4, 8, …}`. Vì `7 = 111₂` chỉ có 3 bit, chỉ `c = 2` có đóng góp: các số `≤ 7` có đúng 2 bit 1 là `3, 5, 6` ⇒ `3`. ✔

## 5. Cách 1 — Brute Force

In [5]:
def brute(n, k):
    return sum(1 for x in range(1, n + 1) if depth(x) == k)

for n, k, exp in examples:
    print(brute(n, k), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


**Nút thắt:** `O(n log n)`.

## 6. Ý tưởng thuật toán / bước đột phá

`answer(n, k) = Σ_{c : depth(c) = k−1} count_with_ones(n, c) − [k = 1]` (trừ `x = 1` khi `c = 1` được tính, tức `k − 1 = depth(1) = 0`). Mỗi `count_with_ones` là `O(50)` ⇒ tổng `O(50²)`.

**Vì sao không cần digit DP phức tạp?** Điều kiện "độ sâu bằng `k`" phụ thuộc vào `x` **chỉ qua** `popcount(x)`. Vậy thay vì một DP mang theo cả dãy lặp, ta tách thành hai tầng: tầng ngoài liệt kê ≤ 50 giá trị `c` hợp lệ (tra bảng độ sâu), tầng trong đếm số `x ≤ n` có đúng `c` bit 1 bằng tổ hợp. Đây là mẫu chung cho mọi bài "hàm lặp co nhanh": chỉ bước đầu tiên tác động lên không gian khổng lồ.

**Kiểm tra tổng:** với mọi `n`, `Σ_{k=0}^{5} popcountDepth(n, k) = n` — một phép thử nhanh rất hữu ích (mục 12.3).

## 7. Các cách tiếp theo

- **Cách 2:** digit DP có nhớ `(vị trí bit, số bit 1 đã đặt, tight)` cho mỗi `c` — cách tổng quát.
- **Cách 3:** tổ hợp theo bit (lời giải nộp).
- **Cách 4:** một lần digit DP đếm **phân phối** popcount của mọi `x ≤ n`, rồi gom theo độ sâu.

### 7.1. Cách 2 — Digit DP có nhớ

In [6]:
def digit_dp(n, k):
    if k == 0: return 1
    bits = bin(n)[2:]

    @lru_cache(None)
    def go(i, ones, tight):
        if i == len(bits): return {ones: 1}
        res = {}
        up = int(bits[i]) if tight else 1
        for d in range(up + 1):
            for c, v in go(i + 1, ones + d, tight and d == up).items():
                res[c] = res.get(c, 0) + v
        return res

    dist = go(0, 0, True)
    total = sum(v for c, v in dist.items() if c >= 1 and depth(c) == k - 1)
    return total - (1 if k == 1 else 0)

for n, k, exp in examples:
    print(digit_dp(n, k), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


### 7.2. Cách 3 — Tổ hợp theo bit (lời giải chính)

In [7]:
DEPTH = [0] * 64
for c in range(2, 64): DEPTH[c] = 1 + DEPTH[bin(c).count("1")]

def combinatorial(n, k):
    if k == 0: return 1
    total = sum(count_with_ones(n, c) for c in range(1, 61) if DEPTH[c] == k - 1)
    return total - (1 if k == 1 else 0)

for n, k, exp in examples:
    print(combinatorial(n, k), "kỳ vọng", exp)

2 kỳ vọng 2
3 kỳ vọng 3


### 7.3. Cách 4 — Phân phối popcount một lần

In [8]:
def popcount_distribution(n):
    return [count_with_ones(n, c) for c in range(n.bit_length() + 1)]

def via_distribution(n, k):
    if k == 0: return 1
    dist = popcount_distribution(n)
    return sum(dist[c] for c in range(1, len(dist)) if DEPTH[c] == k - 1) - (1 if k == 1 else 0)

for n, k, exp in examples:
    print(via_distribution(n, k), "kỳ vọng", exp)
print("phân phối popcount của x ≤ 100:", popcount_distribution(100))

2 kỳ vọng 2
3 kỳ vọng 3
phân phối popcount của x ≤ 100: [1, 7, 21, 33, 26, 11, 2, 0]


## 8. TRACE từng bước

In [9]:
def trace(n, k):
    print(f"n = {n} = {bin(n)[2:]}, k = {k}")
    cs = [c for c in range(1, n.bit_length() + 1) if DEPTH[c] == k - 1]
    print("  các c có depth(c) = k-1:", cs)
    total = 0
    for c in cs:
        res = 0; ones = 0; steps = []
        for b in range(n.bit_length() - 1, -1, -1):
            if n >> b & 1:
                add = comb(b, c - ones) if 0 <= c - ones <= b else 0
                steps.append(f"bit {b}: đặt 0 -> C({b},{c - ones})={add}"); res += add; ones += 1
        res += ones == c
        print(f"  c = {c}: " + "; ".join(steps) + f"; chính n? {ones == c} -> {res}")
        total += res
    if k == 1: total -= 1; print("  trừ x = 1")
    print("  đáp án:", total, "| brute:", brute(n, k))

trace(7, 2)
trace(20, 1)

n = 7 = 111, k = 2
  các c có depth(c) = k-1: [2]
  c = 2: bit 2: đặt 0 -> C(2,2)=1; bit 1: đặt 0 -> C(1,1)=1; bit 0: đặt 0 -> C(0,0)=1; chính n? False -> 3
  đáp án: 3 | brute: 3
n = 20 = 10100, k = 1
  các c có depth(c) = k-1: [1]
  c = 1: bit 4: đặt 0 -> C(4,1)=4; bit 2: đặt 0 -> C(2,0)=1; chính n? False -> 5
  trừ x = 1
  đáp án: 4 | brute: 4


## 9. Code nộp LeetCode

In [10]:
from math import comb

class Solution:
    def popcountDepth(self, n: int, k: int) -> int:
        if k == 0:
            return 1
        depth = [0] * 64
        for c in range(2, 64):
            depth[c] = 1 + depth[bin(c).count('1')]
        L = n.bit_length()
        total = 0
        for c in range(1, L + 1):
            if depth[c] != k - 1:
                continue
            ones = 0
            for b in range(L - 1, -1, -1):
                if n >> b & 1:
                    need = c - ones
                    if 0 <= need <= b:
                        total += comb(b, need)
                    ones += 1
            if ones == c:
                total += 1
        if k == 1:
            total -= 1                      # x = 1 có popcount 1 nhưng độ sâu 0
        return total

sol = Solution()
for n, k, exp in examples:
    got = sol.popcountDepth(n, k)
    print(n, k, "->", got, "✔" if got == exp else "✘")

4 1 -> 2 ✔
7 2 -> 3 ✔


## 10. Vì sao thuật toán đúng

1. **Đệ quy độ sâu:** với `x > 1`, `p_1 = popcount(x)` và dãy từ `p_1` là dãy của `popcount(x)` ⇒ `depth(x) = 1 + depth(popcount(x))`. Với `x = 1`, `depth = 0`.
2. **Phân loại theo popcount:** `{x ∈ [2, n] : depth(x) = k} = ⋃_{c : depth(c) = k−1} {x ∈ [2, n] : popcount(x) = c}` (các tập rời nhau).
3. **Đếm đúng:** `count_with_ones(n, c)` đếm `x ∈ [0, n]` có `c` bit 1: mỗi `x < n` có bit khác nhau đầu tiên tại một bit 1 của `n` (nơi `x` đặt 0), các bit cao hơn trùng `n` (`ones` bit 1), các bit thấp hơn tự do (`C(b, c − ones)` cách). Cộng `n` riêng.
4. Với `c ≥ 1` không đếm `0`; với `c = 1` thì có đếm `x = 1` ⇒ trừ khi `k = 1`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n log n)` | `O(1)` |
| 2. Digit DP | `O(L² · L)` | `O(L²)` |
| 3. Tổ hợp theo bit | `O(L²)` với `L ≤ 50` | `O(1)` |
| 4. Phân phối popcount | `O(L²)` | `O(L)` |

## 12. Bộ test

In [11]:
METHODS = [brute, digit_dp, combinatorial, via_distribution, lambda n, k: Solution().popcountDepth(n, k)]
for n, k, exp in examples:
    assert all(f(n, k) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 mọi n ≤ 3000, mọi k
for n in range(1, 3001):
    for k in range(0, 6):
        b = brute(n, k) if n < 400 else None
        r = Solution().popcountDepth(n, k)
        assert r == combinatorial(n, k) == via_distribution(n, k)
        if b is not None: assert r == b == digit_dp(n, k), (n, k)
# đối chiếu với đếm tích luỹ cho n tới 3000
cnt = [0] * 6
for x in range(1, 3001):
    cnt[depth(x)] += 1
    if x % 500 == 0: assert [Solution().popcountDepth(x, k) for k in range(6)] == cnt
print("n ≤ 3000: OK")

n ≤ 3000: OK


In [13]:
# 12.3 n lớn: các cách nhanh khớp nhau; tổng theo k bằng n
rng = random.Random(3621)
for _ in range(300):
    n = rng.randint(1, 10**15)
    vals = [Solution().popcountDepth(n, k) for k in range(6)]
    assert sum(vals) == n and vals == [via_distribution(n, k) for k in range(6)]
print("300 n lớn: OK | n = 10^15 ->", [Solution().popcountDepth(10**15, k) for k in range(6)])

300 n lớn: OK | n = 10^15 -> [1, 49, 18469940886122, 483140975982382, 498389083131446, 0]


In [14]:
# 12.4 thời gian
t0 = time.time()
for _ in range(2000): Solution().popcountDepth(10**15, 3)
print("2000 lần với n = 10^15:", round(time.time() - t0, 3), "s")

2000 lần với n = 10^15: 0.305 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trừ `x = 1` khi `k = 1`

In [15]:
def wrong_no_sub(n, k):
    if k == 0: return 1
    return sum(count_with_ones(n, c) for c in range(1, 61) if DEPTH[c] == k - 1)        # SAI

print("sai:", wrong_no_sub(4, 1), "| đúng:", brute(4, 1))

sai: 3 | đúng: 2


### Lỗi 2 — Trả `0` cho `k = 0`

In [16]:
def wrong_k0(n, k):
    return 0 if k == 0 else combinatorial(n, k)                                       # SAI

print("sai:", wrong_k0(5, 0), "| đúng:", brute(5, 0))

sai: 0 | đúng: 1


### Lỗi 3 — Định nghĩa độ sâu lệch 1 (đếm cả bước cuối "1 → 1")

In [17]:
def wrong_depth(x):
    d = 1
    while x != 1: x = bin(x).count("1"); d += 1                                        # SAI: bắt đầu từ 1
    return d

print("sai depth(7):", wrong_depth(7), "| đúng:", depth(7))

sai depth(7): 4 | đúng: 3


### Lỗi 4 — Đếm số `≤ n` có `c` bit 1 mà quên cộng chính `n`

In [18]:
def wrong_count(n, c):
    res = 0; ones = 0
    for b in range(n.bit_length() - 1, -1, -1):
        if n >> b & 1:
            if 0 <= c - ones <= b: res += comb(b, c - ones)
            ones += 1
    return res                                                                          # SAI: thiếu n

print("sai:", wrong_count(6, 2), "| đúng:", count_with_ones(6, 2))

sai: 2 | đúng: 3


### Lỗi 5 — Dùng `bin(n).count("1")` của **n** thay vì duyệt mọi `c`

Một số người nhầm "độ sâu của các số ≤ n" với "độ sâu của n". Độ sâu thay đổi thất thường giữa các số liền nhau (`7 → 3` nhưng `8 → 1`), nên phải đếm theo từng `c`.

In [19]:
print("độ sâu của 1..16:", [depth(x) for x in range(1, 17)])
print("đếm k = 1 với n = 8: đúng", Solution().popcountDepth(8, 1), "| chỉ nhìn depth(8) = 1 rồi trả 'mọi số' là sai")

độ sâu của 1..16: [0, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 2, 1]
đếm k = 1 với n = 8: đúng 3 | chỉ nhìn depth(8) = 1 rồi trả 'mọi số' là sai


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Hàm lặp `x → popcount(x)` co rất nhanh (`10^{15} → ≤ 50 → ≤ 5 → …`) ⇒ chỉ bước **đầu tiên** cần kỹ thuật đếm, các bước sau tra bảng.
- "Đếm số `≤ n` có đúng `c` bit 1" ⇒ duyệt bit 1 của `n`, cộng `C(b, c − ones)`.
- Luôn kiểm tra các ca biên `x = 1`, `k = 0`.

## 15. Biến thể của bài

1. **Bản II**: mảng + cập nhật điểm + truy vấn đoạn đếm số phần tử có độ sâu `k` ⇒ 6 Fenwick Tree.
2. **Đếm trên đoạn `[a, b]`**: `f(b) − f(a − 1)`.
3. **Độ sâu theo hàm khác** (tổng chữ số thập phân): digit DP theo tổng chữ số.
4. **Số `x ≤ n` có popcount nguyên tố**: cùng khung, đổi điều kiện trên `c`.

In [20]:
# Biến thể 3: "độ sâu tổng chữ số" — x -> tổng chữ số cho tới khi < 10; đếm số x ≤ n có độ sâu k
def digit_sum_depth(x):
    d = 0
    while x >= 10: x = sum(map(int, str(x))); d += 1
    return d

def count_digit_sum_depth(n, k):
    s = str(n)
    @lru_cache(None)
    def go(i, tot, tight):
        if i == len(s): return {tot: 1}
        res = {}; up = int(s[i]) if tight else 9
        for d in range(up + 1):
            for t, v in go(i + 1, tot + d, tight and d == up).items(): res[t] = res.get(t, 0) + v
        return res
    dist = go(0, 0, True); total = 0
    for t, v in dist.items():
        if t == 0: continue                               # bỏ x = 0
        if t < 10:
            total += v * 0                                # xử lý riêng bên dưới
        elif 1 + digit_sum_depth(t) == k: total += v
    # x có tổng chữ số < 10: độ sâu 0 nếu x < 10, độ sâu 1 nếu x >= 10
    small_total = sum(v for t, v in dist.items() if 1 <= t < 10)
    single = min(n, 9)
    if k == 0: total += single
    if k == 1: total += small_total - single
    return total

for n in range(1, 2000):
    for k in range(4):
        assert count_digit_sum_depth(n, k) == sum(1 for x in range(1, n + 1) if digit_sum_depth(x) == k), (n, k)
print("Biến thể 3 OK")

Biến thể 3 OK


**Biến thể 1 (bản II) — phác thảo.** Mỗi phần tử có độ sâu `d ∈ [0, 5]` (vì `nums[i] ≤ 10^{15}`). Giữ 6 Fenwick Tree `F_d`; truy vấn `[1, l, r, k]` = `F_k.sum(l, r)`; cập nhật `[2, i, v]` = trừ 1 ở `F_{cũ}`, cộng 1 ở `F_{mới}`. Độ sâu của một số tính bằng vòng lặp popcount (`≤ 5` bước). Bài 3624 trong cùng đợt cài đặt đầy đủ ý này.

In [21]:
class DepthCounter:
    def __init__(self, nums):
        self.n = len(nums); self.t = [[0] * (self.n + 1) for _ in range(6)]; self.d = [depth(x) for x in nums]
        for i, dd in enumerate(self.d): self._add(dd, i, 1)
    def _add(self, dd, i, v):
        i += 1
        while i <= self.n: self.t[dd][i] += v; i += i & -i
    def _pre(self, dd, i):
        s = 0
        while i > 0: s += self.t[dd][i]; i -= i & -i
        return s
    def query(self, l, r, k): return self._pre(k, r + 1) - self._pre(k, l) if k < 6 else 0
    def update(self, i, v):
        self._add(self.d[i], i, -1); self.d[i] = depth(v); self._add(self.d[i], i, 1)

dc = DepthCounter([3, 5, 6]); out = [dc.query(0, 2, 2)]; dc.update(1, 4); out += [dc.query(1, 2, 1), dc.query(0, 1, 0)]
print("ví dụ 2 của bản II:", out)
assert out == [3, 1, 0]

ví dụ 2 của bản II: [3, 1, 0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Từng số | `O(n log n)` | Kiểm chứng |
| 2. Digit DP | Trạng thái `(i, ones, tight)` | `O(L³)` | Mẫu tổng quát |
| 3. Tổ hợp theo bit | Đếm trực tiếp | `O(L²)` | **Lời giải chuẩn** |
| 4. Phân phối popcount | Một lần cho mọi `c` | `O(L²)` | Nhiều `k` cùng `n` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `popcountDepth(1, 0)`, `popcountDepth(1, 1)`.

**Bài 2 (tính tay).** `popcountDepth(16, 1)`.

**Bài 3 (điền TODO).** Viết `count_range(a, b, k)`.

**Bài 4 (tự viết lại).** Tính `count_with_ones` không dùng `math.comb` (tam giác Pascal).

**Bài 5 (mini challenge).** Tìm số `x` nhỏ nhất có độ sâu `k` (với `k = 0..5`).

In [22]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_range(a, b, k):
    # TODO
    pass

In [23]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def smallest_with_depth(k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `x = 1` có độ sâu 0. (2) `n = 1`: không có `x` độ sâu 1. (3) `1` và `0`.

**Bài 2.** (1) Độ sâu 1 ⇔ popcount = 1 và `x > 1`. (2) `2, 4, 8, 16`. (3) `4`.

**Bài 3.** (1) `f(b) − f(a − 1)`. (2) `a = 1` ⇒ `f(0) = 0`. (3) Kiểm chứng.

**Bài 4.** (1) Bảng `C[i][j]` tới 64. (2) Thay `comb`. (3) So kết quả.

**Bài 5.** (1) Độ sâu 1: `2`. (2) Độ sâu `k`: số nhỏ nhất có popcount = số nhỏ nhất độ sâu `k−1`, tức `2^{m} − 1` với `m` = đáp án bậc trước. (3) `1, 2, 3, 7, 127, 2^{127}−1`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().popcountDepth(1, 0) == 1 and Solution().popcountDepth(1, 1) == 0
assert Solution().popcountDepth(16, 1) == 4 == brute(16, 1)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [25]:
# Bài 3, 4
def count_range_sol(a, b, k):
    return Solution().popcountDepth(b, k) - (Solution().popcountDepth(a - 1, k) if a > 1 else 0)

C = [[0] * 65 for _ in range(65)]
for i in range(65):
    C[i][0] = 1
    for j in range(1, i + 1): C[i][j] = C[i - 1][j - 1] + C[i - 1][j]

def count_with_ones_pascal(n, c):
    res = 0; ones = 0
    for b in range(n.bit_length() - 1, -1, -1):
        if n >> b & 1:
            if 0 <= c - ones <= b: res += C[b][c - ones]
            ones += 1
    return res + (ones == c)

rng = random.Random(4)
for _ in range(300):
    a = rng.randint(1, 2000); b = rng.randint(a, 2500); k = rng.randint(0, 4)
    assert count_range_sol(a, b, k) == sum(1 for x in range(a, b + 1) if depth(x) == k)
    n = rng.randint(1, 10**15); c = rng.randint(0, 50)
    assert count_with_ones_pascal(n, c) == count_with_ones(n, c)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [26]:
# Bài 5
def smallest_with_depth_sol(k):
    x = 1
    for _ in range(k):
        x = 2 if x == 1 else (1 << x) - 1
    return x

res = [smallest_with_depth_sol(k) for k in range(5)]
assert res[:4] == [min(x for x in range(1, 200) if depth(x) == k) for k in range(4)]
assert all(depth(smallest_with_depth_sol(k)) == k for k in range(5))
print("Bài 5 OK:", res, "| k = 5 có", smallest_with_depth_sol(5).bit_length(), "bit")

Bài 5 OK: [1, 2, 3, 7, 127] | k = 5 có 127 bit


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
if k == 0: return 1
depth[c] = 1 + depth[popcount(c)] (depth[1] = 0)
ans = Σ_{c>=1, depth[c] == k-1} count_with_ones(n, c)
if k == 1: ans -= 1            # bỏ x = 1
# count_with_ones: duyệt bit 1 của n từ cao xuống, cộng C(b, c - ones); +1 nếu popcount(n) == c
```

**Checklist:**

- [ ] `k = 0` ⇒ `1`?
- [ ] Trừ `x = 1` khi `k = 1`?
- [ ] Cộng chính `n` trong `count_with_ones`?
- [ ] Độ sâu của `1` là 0?
- [ ] Tổng theo mọi `k` bằng `n`?